# VideoMAE-v1 — Emotion/Violence Recognition from RGB Video

## Overview
This notebook fine-tunes **VideoMAE-v1** (`MCG-NJU/videomae-base`) on RGB video data for action/emotion recognition.

> **VideoMAE-v1** (Masked Autoencoders for Video) uses a **tube masking** strategy during pre-training. The model expects inputs of shape **(B, T, C, H, W)** where **T = 16** frames.

---

## Dataset Configuration

| Setting | 3-Class Problem | 7-Class Problem |
|---|---|---|
| Classes | NonViolence (0), PreViolence (1), Violence (2) | Pointing (0), CollarGrabbing (1), Pushing (2), Strangling (3), Hitting (4), Headlock (5), KickingAttack (6) |
| `num_labels` | **3** | **7** |
| Use case | Coarse violence detection | Fine-grained behavior recognition |

> **Switch between 3-class and 7-class** by changing `NUM_CLASSES` and `LABEL_MAP` in the *Configuration* cell below.

---

## Input Data Format (RGB.npy)

Each video is stored as a `.npy` file with shape **(T, H, W, C)**:
- `T` — number of frames (variable per video, will be sampled to 16)
- `H` — frame height (224 px)
- `W` — frame width (224 px)
- `C` — channels (3, RGB)

Files are organized under `FEATURE_ROOT/` mirroring the dataset structure:
```
FEATURE_ROOT/
  <Class>/
    <Scene>/
      <Angle>/
        [<Behavior>/]   # only for Violence / PreViolence
          <video_name>.npy
```

## CSV Format
```
video,class
Violence/scene01/Top/Hitting/v001,Hitting
NonViolence/scene02/Center/v012,NonViolence
```

## Part 1 — Imports & Installation

In [ ]:
# Install required packages
!pip install -q transformers accelerate

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
from transformers import VideoMAEForVideoClassification

## Part 2 — Configuration

Change `NUM_CLASSES` and `LABEL_MAP` to switch between **3-class** and **7-class** settings.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Root directory containing .npy video files (shape: T, H, W, C)
FEATURE_ROOT = "/content/drive/MyDrive/EarlyViolenceDetection2025/Features/Video_NPY"

# Number of frames expected by VideoMAE-v1
TARGET_FRAMES = 16   # VideoMAE-v1 is pre-trained with 16-frame clips

# ----- 3-Class Setup -----
# Uncomment this block for coarse violence detection
NUM_CLASSES = 3
LABEL_MAP = {
    "NonViolence": 0,
    "PreViolence": 1,
    "Violence":    2,
}

# ----- 7-Class Setup -----
# Uncomment this block for fine-grained behavior recognition
# NUM_CLASSES = 7
# LABEL_MAP = {
#     "Pointing":       0,
#     "CollarGrabbing": 1,
#     "Pushing":        2,
#     "Strangling":     3,
#     "Hitting":        4,
#     "Headlock":       5,
#     "KickingAttack":  6,
# }

# Training hyperparameters
BATCH_SIZE      = 16
LEARNING_RATE   = 1e-5
WEIGHT_DECAY    = 1e-4
EPOCHS          = 40
PATIENCE        = 5     # early stopping patience (based on val loss)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print(f"Task: {NUM_CLASSES}-class classification")
print("Label map:", LABEL_MAP)

## Part 3 — Dataset

`VideoDataset` loads each `.npy` file and:
1. **Samples** `TARGET_FRAMES` frames uniformly (or zero-pads if the video is shorter).
2. Normalizes pixel values to `[0, 1]`.
3. Returns a tensor of shape `(C, T, H, W)` — which is then permuted to `(T, C, H, W)` before feeding to VideoMAE.

In [ ]:
class VideoDataset(Dataset):
    """
    Loads RGB video clips stored as .npy files.

    Each .npy file has shape (T, H, W, C) where:
      T = number of frames (variable)
      H = W = 224  (frame resolution)
      C = 3        (RGB channels)

    The dataset searches for the matching .npy file by walking
    FEATURE_ROOT/<Class>/ so the directory can be arbitrarily nested.
    """

    def __init__(self, csv_path, root):
        self.df   = pd.read_csv(csv_path)
        self.root = root

    def find_feature(self, video_rel):
        """Locate the .npy file for a video given its relative path."""
        video_name = os.path.basename(video_rel)
        npy_name   = os.path.splitext(video_name)[0] + ".npy"
        cls        = video_rel.split("/")[0]
        search_root = os.path.join(self.root, cls)

        for r, _, files in os.walk(search_root):
            if npy_name in files:
                return os.path.join(r, npy_name)
        return None

    def sample_frames(self, video):
        """
        Normalize frame count to TARGET_FRAMES.
        - If T >= TARGET_FRAMES: uniform sampling.
        - If T <  TARGET_FRAMES: zero-pad at the end.

        Args:
            video (np.ndarray): shape (T, H, W, C)
        Returns:
            np.ndarray: shape (TARGET_FRAMES, H, W, C)
        """
        T = video.shape[0]

        if T >= TARGET_FRAMES:
            idx   = np.linspace(0, T - 1, TARGET_FRAMES).astype(int)
            video = video[idx]
        else:
            pad   = np.zeros(
                (TARGET_FRAMES - T, video.shape[1], video.shape[2], video.shape[3]),
                dtype=video.dtype
            )
            video = np.concatenate([video, pad], axis=0)

        return video

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        video_rel = row["video"]
        label     = LABEL_MAP[row["class"]]

        npy_path = self.find_feature(video_rel)
        video    = np.load(npy_path)           # (T, H, W, C)
        video    = self.sample_frames(video)   # (16, H, W, C)
        video    = video.astype(np.float32) / 255.0
        video    = torch.tensor(video).permute(3, 0, 1, 2)  # (C, T, H, W)

        return video, label

## Part 4 — DataLoaders

In [ ]:
train_dataset = VideoDataset("train.csv", FEATURE_ROOT)
val_dataset   = VideoDataset("val.csv",   FEATURE_ROOT)
test_dataset  = VideoDataset("test.csv",  FEATURE_ROOT)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,  shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} samples")
print(f"Val  : {len(val_dataset)} samples")
print(f"Test : {len(test_dataset)} samples")

## Part 5 — Model

`VideoMAEForVideoClassification` from HuggingFace wraps the VideoMAE-v1 backbone
with a linear classification head.

**Input shape expected by VideoMAE-v1:** `(B, T, C, H, W)` where `T = 16`.

In this notebook, the DataLoader yields `(C, T, H, W)`, so we permute to
`(B, T, C, H, W)` inside the training loop via `video.permute(0, 2, 1, 3, 4)`.

In [ ]:
model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,   # replace the pre-trained head
)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

criterion = nn.CrossEntropyLoss()

print("Model loaded:", type(model).__name__)
print(f"Output classes: {NUM_CLASSES}")

## Part 6 — Training Loop

Early stopping is applied when validation loss does not improve for `PATIENCE` consecutive epochs.
The best checkpoint is saved as `best_videomae_v1.pth`.

In [ ]:
best_val_loss    = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):

    # ------------------------------------------------------------------ Train
    model.train()
    train_loss  = 0
    y_true_tr, y_pred_tr = [], []

    for video, label in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        # video: (B, C, T, H, W) → VideoMAE expects (B, T, C, H, W)
        video = video.permute(0, 2, 1, 3, 4).to(DEVICE)
        label = label.to(DEVICE)

        outputs = model(pixel_values=video, labels=label)
        loss    = outputs.loss
        logits  = outputs.logits

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = logits.argmax(dim=1)
        y_true_tr.extend(label.cpu().numpy())
        y_pred_tr.extend(preds.cpu().numpy())

    train_loss /= len(train_loader)
    train_acc   = accuracy_score(y_true_tr, y_pred_tr)

    # ---------------------------------------------------------------- Validate
    model.eval()
    val_loss = 0
    y_true_val, y_pred_val = [], []

    with torch.no_grad():
        for video, label in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            video = video.permute(0, 2, 1, 3, 4).to(DEVICE)
            label = label.to(DEVICE)

            outputs  = model(pixel_values=video, labels=label)
            val_loss += outputs.loss.item()
            preds    = outputs.logits.argmax(dim=1)

            y_true_val.extend(label.cpu().numpy())
            y_pred_val.extend(preds.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc   = accuracy_score(y_true_val, y_pred_val)

    print(f"\nEpoch {epoch+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%"
          f" | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    # ---------------------------------------------------------- Early stopping
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_videomae_v1.pth")
        print("  ✓ Best model saved.")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("  Early stopping triggered.")
            break

## Part 7 — Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load("best_videomae_v1.pth", map_location=DEVICE))
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for video, label in tqdm(test_loader, desc="Testing"):
        video = video.permute(0, 2, 1, 3, 4).to(DEVICE)
        label = label.to(DEVICE)

        outputs = model(pixel_values=video)
        preds   = outputs.logits.argmax(dim=1)

        y_true.extend(label.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

acc       = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall    = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1        = f1_score(y_true, y_pred, average="macro", zero_division=0)
cm        = confusion_matrix(y_true, y_pred)

print("\n===== TEST RESULTS =====")
print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall   : {recall*100:.2f}%")
print(f"F1-score : {f1*100:.2f}%")
print("\nConfusion Matrix")
print(cm)